# Практическая работа 2. Группировка и объединение таблиц

Вариант 8

In [1]:
import numpy as np
import pandas as pd

sales = pd.DataFrame({
    "OrderID": range(101, 111),
    "Product": ["Laptop", "Phone", "Tablet", "Laptop", "Phone", "Laptop", "Tablet", "Phone", "Laptop", "Tablet"],
    "Quantity": [1, 2, 1, 3, 1, 2, 1, 4, 1, 2],
    "Price": [50000, 30000, 20000, 50000, np.nan, 50000, 20000, 30000, 50000, np.nan],
    "Region": ["Moscow", "SPb", "Kazan", "Moscow", "SPb", "Kazan", np.nan, "SPb", "Moscow", "Kazan"],
})
products = pd.DataFrame({
    "Product": ["Laptop", "Phone", "Tablet", "Monitor"],
    "Category": ["Electronics", "Electronics", "Electronics", "Accessories"],
    "Weight_kg": [2.5, 0.2, 0.5, 3.0],
})

### 1. Три заказа с самой высокой ценой за единицу

При одинаковой цене берём первые строки в исходном порядке.

In [2]:
sales.nlargest(3, "Price")

,OrderID,Product,Quantity,Price,Region
0,101,Laptop,1,50000.0,Moscow
3,104,Laptop,3,50000.0,Moscow
5,106,Laptop,2,50000.0,Kazan


### 2. Выручка по товару и региону

Суммируем произведения цены и количества. Строку без региона сохраняем отдельной группой. Группа, в которой неизвестны все цены, получит пропуск.

In [3]:
amounts = sales.assign(Revenue=sales["Quantity"] * sales["Price"])
amounts.groupby(["Product", "Region"], dropna=False)["Revenue"].sum(min_count=1)

Product  Region
Laptop   Kazan     100000.0
         Moscow    250000.0
Phone    SPb       180000.0
Tablet   Kazan      20000.0
         NaN        20000.0
Name: Revenue, dtype: float64

### 3. Замена пропусков нулём

In [4]:
filled = sales.fillna(0)
filled

,OrderID,Product,Quantity,Price,Region
0,101,Laptop,1,50000.0,Moscow
1,102,Phone,2,30000.0,SPb
2,103,Tablet,1,20000.0,Kazan
3,104,Laptop,3,50000.0,Moscow
4,105,Phone,1,0.0,SPb
5,106,Laptop,2,50000.0,Kazan
6,107,Tablet,1,20000.0,0
7,108,Phone,4,30000.0,SPb
8,109,Laptop,1,50000.0,Moscow
9,110,Tablet,2,0.0,Kazan


### 4. Сортировка по полной стоимости заказа

In [5]:
filled["Total"] = filled["Quantity"] * filled["Price"]
filled.sort_values("Total", ascending=False, kind="stable")

,OrderID,Product,Quantity,Price,Region,Total
3,104,Laptop,3,50000.0,Moscow,150000.0
7,108,Phone,4,30000.0,SPb,120000.0
5,106,Laptop,2,50000.0,Kazan,100000.0
1,102,Phone,2,30000.0,SPb,60000.0
0,101,Laptop,1,50000.0,Moscow,50000.0
8,109,Laptop,1,50000.0,Moscow,50000.0
2,103,Tablet,1,20000.0,Kazan,20000.0
6,107,Tablet,1,20000.0,0,20000.0
4,105,Phone,1,0.0,SPb,0.0
9,110,Tablet,2,0.0,Kazan,0.0


### 5. Категория товара в каждом заказе

In [6]:
joined = sales.merge(products, on="Product", how="inner", validate="many_to_one")
joined[["OrderID", "Product", "Category"]]

,OrderID,Product,Category
0,101,Laptop,Electronics
1,102,Phone,Electronics
2,103,Tablet,Electronics
3,104,Laptop,Electronics
4,105,Phone,Electronics
5,106,Laptop,Electronics
6,107,Tablet,Electronics
7,108,Phone,Electronics
8,109,Laptop,Electronics
9,110,Tablet,Electronics
